In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Build City Reference from config.json

In [0]:
df_cities_raw = spark.read.option('multiline', 'true').json('abfss://bronze@logisticdatalakestorage.dfs.core.windows.net/config/config.json')

df_city_ref = df_cities_raw \
    .select(explode(col('cities')).alias('c')) \
    .select(
        initcap(col('c.name')).alias('mapped_city'),
        upper(col('c.country')).alias('mapped_country'),
        col('c.lat').alias('c_lat'),
        col('c.lon').alias('c_lon')
    )

print(f'City reference rows: {df_city_ref.count()}')
df_city_ref.display()

City reference rows: 10


mapped_city,mapped_country,c_lat,c_lon
Mumbai,IN,19.076,72.8777
Berlin,DE,52.52,13.405
London,GB,51.5074,-0.1278
New York,US,40.7128,-74.006
Bangkok,TH,13.7563,100.5018
Dubai,AE,25.2048,55.2708
Singapore,SG,1.3521,103.8198
Sydney,AU,-33.8688,151.2093
Paris,FR,48.8566,2.3522
Sao Paulo,BR,-23.5505,-46.6333


### Read Bronze JSON files

In [0]:
bronze_path = "abfss://bronze@logisticdatalakestorage.dfs.core.windows.net/tomtom/"
silver_path = "abfss://silver@logisticdatalakestorage.dfs.core.windows.net/tomtom/"

In [0]:
df_raw = spark.read.format('json').load(bronze_path)
df_raw.limit(1).display()

incidents,year,month,day,hour


In [0]:
df_raw.printSchema()

root
 |-- incidents: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- geometry: struct (nullable = true)
 |    |    |    |-- coordinates: array (nullable = true)
 |    |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- type: string (nullable = true)
 |    |    |-- properties: struct (nullable = true)
 |    |    |    |-- iconCategory: long (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)



### Explode incidents array

In [0]:
df_exploded = df_raw.withColumn('incident', explode(col('incidents'))) \
    .drop('incidents')

### Flatten incident fields

In [0]:
df_flat = df_exploded.select(
    col('year'), col('month'), col('day'), col('hour'),
    col('incident.type').alias('incident_type'),
    col('incident.properties.iconCategory').alias('icon_category'),
    col('incident.geometry.type').alias('geometry_type'),
    # [0] works because coordinates is ARRAY — element is STRING
    col('incident.geometry.coordinates')[0][0].alias('lon'),
    col('incident.geometry.coordinates')[0][1].alias('lat')
)

print(f'Incidents extracted: {df_flat.count()}')
df_flat.limit(5).display()

Incidents extracted: 3275


year,month,day,hour,incident_type,icon_category,geometry_type,lon,lat
2026,5,27,12,Feature,6,LineString,-46.7366271249,-23.5086715128
2026,5,27,12,Feature,6,LineString,-46.7350982657,-23.6333324786
2026,5,27,12,Feature,6,LineString,-46.7326212457,-23.5436810466
2026,5,27,12,Feature,6,LineString,-46.7339690558,-23.5072726933
2026,5,27,12,Feature,8,LineString,-46.7331308654,-23.5077997407


In [0]:
df_flat.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- icon_category: long (nullable = true)
 |-- geometry_type: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lat: double (nullable = true)



### Build event_time from Partition Columns

In [0]:
df_flat = df_flat.withColumn(
    'event_time',
    to_timestamp(
        concat(
            col('year').cast('string'), lit('-'),
            lpad(col('month').cast('string'), 2, '0'), lit('-'),
            lpad(col('day').cast('string'), 2, '0'), lit(' '),
            lpad(col('hour').cast('string'), 2, '0'), lit(':00:00')
        ),
        'yyyy-MM-dd HH:mm:ss'
    )
)

df_flat.limit(3).display()

year,month,day,hour,incident_type,icon_category,geometry_type,lon,lat,event_time
2026,5,27,12,Feature,6,LineString,-46.7366271249,-23.5086715128,2026-05-27T12:00:00.000Z
2026,5,27,12,Feature,6,LineString,-46.7350982657,-23.6333324786,2026-05-27T12:00:00.000Z
2026,5,27,12,Feature,6,LineString,-46.7326212457,-23.5436810466,2026-05-27T12:00:00.000Z


### Extract Representative lat/lon from Coordinates

In [0]:
df_flat.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- icon_category: long (nullable = true)
 |-- geometry_type: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



### Drop rows where coordinates could not be parsed

In [0]:
df_flat = df_flat.filter(
    col('lat').isNotNull() & col('lon').isNotNull()
)

print(f'Rows with valid coordinates: {df_flat.count()}')
df_flat.limit(5).display()

Rows with valid coordinates: 3275


year,month,day,hour,incident_type,icon_category,geometry_type,lon,lat,event_time
2026,5,27,12,Feature,6,LineString,-46.7366271249,-23.5086715128,2026-05-27T12:00:00.000Z
2026,5,27,12,Feature,6,LineString,-46.7350982657,-23.6333324786,2026-05-27T12:00:00.000Z
2026,5,27,12,Feature,6,LineString,-46.7326212457,-23.5436810466,2026-05-27T12:00:00.000Z
2026,5,27,12,Feature,6,LineString,-46.7339690558,-23.5072726933,2026-05-27T12:00:00.000Z
2026,5,27,12,Feature,8,LineString,-46.7331308654,-23.5077997407,2026-05-27T12:00:00.000Z


### Map Coordinates → City Name

In [0]:
window_city = Window.partitionBy('lat', 'lon', 'event_time').orderBy('_dist')

df_mapped = df_flat \
    .crossJoin(broadcast(df_city_ref)) \
    .withColumn('_lat_diff', abs(col('lat') - col('c_lat'))) \
    .withColumn('_lon_diff', abs(col('lon') - col('c_lon'))) \
    .withColumn('_dist', col('_lat_diff') + col('_lon_diff')) \
    .filter(col('_dist') < 1.5) \
    .withColumn('_rn', row_number().over(window_city)) \
    .filter(col('_rn') == 1) \
    .drop('_lat_diff', '_lon_diff', '_dist', '_rn', 'c_lat', 'c_lon', 'tt_lat', 'tt_lon') \
    .withColumnRenamed('mapped_city', 'city') \
    .withColumnRenamed('mapped_country', 'country_code')

print(f'Rows after city mapping: {df_mapped.count()}')
df_mapped.select('city', 'country_code', 'event_time', 'icon_category').distinct().show()

Rows after city mapping: 3236
+---------+------------+-------------------+-------------+
|     city|country_code|         event_time|icon_category|
+---------+------------+-------------------+-------------+
|   Sydney|          AU|2026-05-27 12:00:00|            7|
|   Sydney|          AU|2026-05-27 12:00:00|            8|
|   Sydney|          AU|2026-05-27 12:00:00|            6|
|   Sydney|          AU|2026-05-27 12:00:00|            0|
|   Sydney|          AU|2026-05-27 12:00:00|            3|
|   Sydney|          AU|2026-05-27 12:00:00|            9|
|Sao Paulo|          BR|2026-05-27 12:00:00|            6|
|Sao Paulo|          BR|2026-05-27 12:00:00|            8|
|Sao Paulo|          BR|2026-05-27 12:00:00|            9|
|Singapore|          SG|2026-05-27 12:00:00|            6|
|Singapore|          SG|2026-05-27 12:00:00|            8|
|Singapore|          SG|2026-05-27 12:00:00|            9|
|  Bangkok|          TH|2026-05-27 12:00:00|            6|
|  Bangkok|          TH|20

In [0]:
df_silver = df_mapped \
    .withColumn('congestion_level',
        when(col('icon_category') == 7, 10)
        .when(col('icon_category') == 6, 8)
        .when(col('icon_category') == 8, 7)
        .when(col('icon_category') == 1, 6)
        .when(col('icon_category') == 3, 5)
        .when(col('icon_category') == 5, 5)
        .when(col('icon_category') == 2, 4)
        .when(col('icon_category') == 4, 4)
        .otherwise(1)
    ) \
    .withColumn('join_time', date_trunc('hour', col('event_time')))

print(f'Final silver rows: {df_silver.count()}')
df_silver.select('city', 'country_code', 'join_time', 'icon_category', 'congestion_level').show(10)

Final silver rows: 3236
+------+------------+-------------------+-------------+----------------+
|  city|country_code|          join_time|icon_category|congestion_level|
+------+------------+-------------------+-------------+----------------+
|Sydney|          AU|2026-05-27 12:00:00|            7|              10|
|Sydney|          AU|2026-05-27 12:00:00|            7|              10|
|Sydney|          AU|2026-05-27 12:00:00|            8|               7|
|Sydney|          AU|2026-05-27 12:00:00|            8|               7|
|Sydney|          AU|2026-05-27 12:00:00|            6|               8|
|Sydney|          AU|2026-05-27 12:00:00|            8|               7|
|Sydney|          AU|2026-05-27 12:00:00|            8|               7|
|Sydney|          AU|2026-05-27 12:00:00|            8|               7|
|Sydney|          AU|2026-05-27 12:00:00|            8|               7|
|Sydney|          AU|2026-05-27 12:00:00|            8|               7|
+------+------------+------

In [0]:
print('Distinct cities mapped:')
df_silver.select('city', 'country_code').distinct().orderBy('city').show()

print('Congestion level distribution:')
df_silver.groupBy('icon_category', 'congestion_level').count().orderBy('congestion_level').show()

print('Schema:')
df_silver.printSchema()

Distinct cities mapped:
+---------+------------+
|     city|country_code|
+---------+------------+
|  Bangkok|          TH|
|   Berlin|          DE|
|    Dubai|          AE|
|   London|          GB|
|   Mumbai|          IN|
| New York|          US|
|    Paris|          FR|
|Sao Paulo|          BR|
|Singapore|          SG|
|   Sydney|          AU|
+---------+------------+

Congestion level distribution:
+-------------+----------------+-----+
|icon_category|congestion_level|count|
+-------------+----------------+-----+
|            9|               1|  205|
|           14|               1|    3|
|            0|               1|   71|
|            3|               5|    7|
|            8|               7|  948|
|            6|               8| 1897|
|            7|              10|  105|
+-------------+----------------+-----+

Schema:
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-

### Write to Silver Delta Lake

In [0]:
df_silver.write.format('delta').mode('overwrite').partitionBy("year", "month", "day").save(silver_path)